<!-- OPS:MAT-02:inicio -->
## Esto es “completo” hoy (55 min)

Marca con lo que realmente ocurrió. No hace falta terminar todo.

- [ ] El entorno funciona o abrí `la ruta de rescate docente (no incluida)` y lo declaré.
- [ ] Producto mínimo: Un fold OOF con pseudo-resultado fuera de muestra; el ATE final puede quedar para S10.
- [ ] Si algo falló: una frase del error, no el traceback completo.
- [ ] Bitácora escrita, aunque diga `lab incompleto`.

Completo ≠ entendido del todo. Incompleto ≠ no puedes seguir. Si la instalación falla después de 10 minutos, usa `la ruta de rescate docente (no incluida)`, conserva el artefacto y marca incompleto.
**Extensión:** Si queda tiempo: cinco folds, varios modelos auxiliares y simulaciones de cobertura.
<!-- OPS:MAT-02:fin -->

# L09 · Ortogonalidad y cross-fitting

**S9 · Lectura: Cap. 10 · 40–55 min**

## La pregunta

¿Cuánto afecta al efecto un error del modelo auxiliar?

## Qué harás

Construir residuos fuera de muestra y entender el score ortogonal.

- Lee la idea breve.
- Ejecuta la demostración y mira el gráfico.
- Completa las celdas marcadas.
- Escribe qué significa el resultado y qué no permite afirmar.

**Producto:** Residuos OOF y una estimación partialling-out.

> **Idea clave:** El cross-fitting evita evaluar el nuisance sobre las mismas observaciones con las que se ajustó.

La solución está en `la ruta de rescate docente (no incluida)`. Intenta primero el notebook del estudiante.

In [ ]:
# Localiza lab_common.py (carpeta actual o una superior)
import sys, pathlib
for _p in [pathlib.Path.cwd(), *list(pathlib.Path.cwd().parents)[:3]]:
    if (_p / "lab_common.py").exists():
        sys.path.insert(0, str(_p)); break

## 0. Preparación

In [ ]:
from lab_common import *
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.linear_model import Lasso, LinearRegression
rng = setup()
N_REP = R(20, 200)
print("réplicas Monte Carlo:", N_REP)

## 1. El modelo y el score (§10.1–10.2)
**PLR:** $Y=\theta D+g(X)+U$, $\;D=m(X)+V$. Queremos $\theta$ sin conocer $g$ ni $m$ (los *nuisance*).

Dos formas de escribir una ecuación de estimación:
- **Plug-in (no ortogonal):** $\hat\theta=\dfrac{\sum D\,(Y-\hat g(X))}{\sum D^2}$ — usa solo $\hat g$.
- **Robinson (ortogonal):** $\hat\theta=\dfrac{\sum (D-\hat m(X))(Y-\hat g_Y(X))}{\sum (D-\hat m(X))^2}$ — **quita $X$ de ambos** lados ($\hat g_Y=E[Y|X]$).

Mundo simulado (el del Listado de la §10): nuisance **no lineales**, $\theta=2$.

In [ ]:
def make_dgp(n=800, p=8, rng=None, theta=2.0):
    rng = rng or np.random.default_rng(0)
    X = rng.normal(size=(n, p))
    m = 0.8 * np.sin(X[:, 0]) + 0.5 * X[:, 1]
    g = np.cos(X[:, 0]) + 0.7 * X[:, 1] + 0.3 * X[:, 2] ** 2
    D = m + rng.normal(size=n)
    Y = theta * D + g + rng.normal(size=n)
    return Y, D, X
make_rf = lambda: ExtraTreesRegressor(n_estimators=40, min_samples_leaf=5, max_features=0.8, random_state=0)   # learner flexible (ExtraTrees)
Y, D, X = make_dgp(800, 8, np.random.default_rng(1))
print("θ verdadero = 2.0 · OLS de Y sobre D y X (mal especificado):", round(ols(Y, np.column_stack([D, X]))["coef"][1], 3))

## 2. Ortogonalidad: lo que *significa* (§10.2)
Un score es **ortogonal** si su sensibilidad a un error pequeño en el nuisance es de **segundo orden**. Lo comprobamos moviendo los nuisance (los verdaderos más una perturbación $\delta\cdot h(X)$) y mirando cuánto se mueve $\hat\theta$.

In [ ]:
m_ver = 0.8 * np.sin(X[:, 0]) + 0.5 * X[:, 1]
g_ver = np.cos(X[:, 0]) + 0.7 * X[:, 1] + 0.3 * X[:, 2] ** 2
gY_ver = 2.0 * m_ver + g_ver                 # E[Y|X]
h = np.sin(X[:, 3]) + X[:, 0]; h = (h - h.mean()) / h.std()        # dirección de error arbitraria, de desviación 1
def theta_plug(dl): return (D * (Y - (g_ver + dl * h))).sum() / (D * D).sum()
def theta_orto(dl): return ((D - (m_ver + dl * h)) * (Y - (gY_ver + dl * h))).sum() / ((D - (m_ver + dl * h)) ** 2).sum()
deltas = np.linspace(-0.3, 0.3, 25)
plt.plot(deltas, [theta_plug(x) for x in deltas], label="plug-in (no ortogonal)"); plt.plot(deltas, [theta_orto(x) for x in deltas], label="Robinson (ortogonal)")
plt.axhline(2, c="gray", ls="--"); plt.xlabel("error δ en el nuisance (en desviaciones de h)"); plt.ylabel("θ̂"); plt.legend(); plt.show()
eps = 0.02
pend = lambda f: (f(eps) - f(-eps)) / (2 * eps)
print(f"pendiente de θ̂ respecto al error del nuisance en δ=0:  plug-in {pend(theta_plug):+.3f} · ortogonal {pend(theta_orto):+.3f}")

**Mira el resultado.** ¿Qué ventaja práctica tiene que la curva ortogonal sea plana cerca de δ = 0?

**Respuesta:**

Que un nuisance **estimado con error** (inevitable con ML flexible) apenas afecta a $\hat\theta$: el error entra de segundo orden. Con el score no ortogonal, el error del nuisance se transmite **directamente** a θ̂.



## 3. El doble sesgo (§10.3): regularización y sobreajuste
Con ML flexible hay **dos** fuentes de sesgo: (1) *regularización* (el nuisance estimado está sesgado) y (2) *sobreajuste* (si el nuisance se ajusta **con las mismas filas** que luego se residualizan, los residuos son demasiado pequeños). El **cross-fitting** ataca (2): cada residuo se calcula con un modelo entrenado en **otras** filas.

### Cross-fitting a mano
`dml_plr_manual(y, d, X, make_learner, K)` debe: partir las filas en $K$ pliegues; para cada pliegue entrenar `Y~X` y `D~X` con las **otras** filas y predecir en ese pliegue; calcular $\hat\theta$ y su error estándar con la función de influencia $\psi=(\tilde y-\hat\theta\tilde d)\,\tilde d/\overline{\tilde d^{\,2}}$, $\;\mathrm{se}=\sqrt{\overline{\psi^2}/n}$.

In [ ]:
# ✏️ Completa tú — dml_plr_manual
# Pista: Residuo = valor observado − predicción de un modelo que NO vio esa fila.
def dml_plr_manual(y, d, X, make_learner, K=5, seed=SEED):
    """Devuelve dict(theta, se, ry, rd). `kfold(n, K, seed)` da los índices de cada pliegue."""
    n = len(y); ry = np.zeros(n); rd = np.zeros(n)
    for te in kfold(n, K, seed):
        tr = np.setdiff1d(np.arange(n), te)
        # TODO: entrena en tr, predice en te, guarda el residuo en ry[te] y rd[te]
        pass
    raise NotImplementedError("completa el bucle de cross-fitting y calcula theta y se")

In [ ]:
res = dml_plr_manual(Y, D, X, make_rf, K=5)
print(f"DML (K=5, ExtraTrees): θ̂ = {fmt_ci(res['theta'], res['se'])}   (θ = 2)")
# Comprobación: cada modelo debe entrenarse con una fracción < 1 de las filas
tam = []
class Rastreador:
    def __init__(self): self.m = LinearRegression()
    def fit(self, A, b): tam.append(len(A)); self.m.fit(A, b); return self
    def predict(self, A): return self.m.predict(A)
_ = dml_plr_manual(Y, D, X, Rastreador, K=5)
print("filas de entrenamiento por ajuste:", sorted(set(tam)), "de", len(Y))
assert max(tam) < len(Y), "¡estás entrenando con todas las filas! eso no es cross-fitting"
assert len(tam) == 10, "debe haber 2 ajustes (Y y D) por pliegue"
print("✔ cross-fitting correcto")

## 4. Monte Carlo: ¿quién sesga y quién cubre? (ejercicio del capítulo)
Comparamos tres estimadores sobre muchos conjuntos de datos: **plug-in** (no ortogonal, Lasso lineal para $g$), **ortogonal sin cross-fitting** (residuos *dentro de muestra* con el mismo learner) y **DML** (ortogonal + cross-fitting).

In [ ]:
def plugin_lasso(y, d, X):
    g = Lasso(alpha=0.05).fit(X, y).predict(X); return (d * (y - g)).sum() / (d * d).sum()
def ortogonal_insample(y, d, X):
    ry = y - make_rf().fit(X, y).predict(X); rd = d - make_rf().fit(X, d).predict(X)
    return (rd * ry).sum() / (rd * rd).sum()
r = np.random.default_rng(0); filas = []
for _ in range(N_REP):
    y_, d_, X_ = make_dgp(800, 8, r)
    o = dml_plr_manual(y_, d_, X_, make_rf, K=5)
    filas.append((plugin_lasso(y_, d_, X_), ortogonal_insample(y_, d_, X_), o["theta"], abs(o["theta"] - 2) < 1.96 * o["se"]))
mc = pd.DataFrame(filas, columns=["plug-in (no ortogonal)", "ortogonal sin cross-fitting", "DML (cross-fitting)", "cubre"])
tab = pd.DataFrame({"sesgo": mc.iloc[:, :3].mean() - 2, "desv. estándar": mc.iloc[:, :3].std()})
show(tab.round(3)); print(f"cobertura del IC 95 % de DML: {mc['cubre'].mean():.2f}  ({N_REP} réplicas)")

In [ ]:
plt.boxplot([mc.iloc[:, k] - 2 for k in range(3)]); plt.xticks([1, 2, 3], ["plug-in", "ortog.\nsin CF", "DML"]); plt.axhline(0, c="gray", ls="--")
plt.ylabel("θ̂ − θ"); plt.title("Distribución del error"); plt.show()

**Mira el resultado.** Ordena los tres por sesgo. ¿Por qué el plug-in sesga *aunque* use un buen modelo para $g$?

**Respuesta:**

El plug-in sesga mucho porque $g$ es no lineal y el Lasso lineal lo aproxima mal; ese error se transmite directo a θ̂ (no ortogonal). El ortogonal sin cross-fitting corrige casi todo, pero sus residuos se calculan con filas que el modelo ya vio: ese sesgo por sobreajuste crece con learners más flexibles (prueba `min_samples_leaf=1`). DML combina ambas protecciones.



## 5. Sobre datos del curso (A2)
IHDP-medium: PLR con DML (K=5, ExtraTrees) frente a OLS.

In [ ]:
sc = load_scenario("synthetic/inference/ihdp_medium"); sc.card()
Xs, ys, ts = sc.xyt("train")
make_rf2 = lambda: ExtraTreesRegressor(n_estimators=60, min_samples_leaf=5, max_features=0.8, random_state=0)
o = dml_plr_manual(ys, ts, Xs, make_rf2, K=5)
r_ols = ols(ys, np.column_stack([ts, Xs])); b_ols, s_ols = r_ols["coef"][1], r_ols["se"][1]
print(f"DML-PLR : {fmt_ci(o['theta'], o['se'])}")
print(f"OLS     : {fmt_ci(b_ols, s_ols)}")
r2_y = 1 - (o["ry"] ** 2).sum() / ((ys - ys.mean()) ** 2).sum(); r2_d = 1 - (o["rd"] ** 2).sum() / ((ts - ts.mean()) ** 2).sum()
print(f"R² fuera de muestra del nuisance: Y~X = {r2_y:.3f}, T~X = {r2_d:.3f}")

In [ ]:
# Vista rápida de los datos del escenario
import matplotlib.pyplot as plt
import numpy as np
_fig, _ax = plt.subplots(1, 2, figsize=(9, 3.2))
try:
    _scenario = globals().get("sc") or globals().get("lc")
    if _scenario is None:
        raise NameError("No hay un escenario cargado para el gráfico editorial")
    _df = _scenario.train
    if _scenario.treatment is not None and _scenario.outcome is not None:
        _t = _df[_scenario.treatment].to_numpy(float)
        _y = _df[_scenario.outcome].to_numpy(float)
        if np.unique(_t).size == 2:
            _counts = [int((_t == value).sum()) for value in (0, 1)]
            _ax[0].bar(["control", "tratado"], _counts)
            _ax[0].set_title("Asignación")
            _ax[1].boxplot([_y[_t == 0], _y[_t == 1]], labels=["control", "tratado"])
            _ax[1].set_title("Resultado observado")
        else:
            _ax[0].hist(_t, bins=20)
            _ax[0].set_title(_scenario.treatment)
            _ax[1].scatter(_t, _y, s=8, alpha=.35)
            _ax[1].set_title(f"{_scenario.treatment} frente a {_scenario.outcome}")
    elif _scenario.adj is not None:
        _ax[0].imshow(_scenario.adj, cmap="Greys", interpolation="nearest")
        _ax[0].set_title("Grafo del escenario")
        _ax[1].axis("off")
    else:
        _ax[0].axis("off"); _ax[1].axis("off")
    _fig.suptitle(f"Escenario: {_scenario.name}")
    _fig.tight_layout(); plt.show()
finally:
    plt.close(_fig)

In [ ]:
# EVALUATION-CELL
print("ATE verdadero del escenario:", None if sc.true_ate is None else round(sc.true_ate, 3))

## 6. Supuesto y bitácora

In [ ]:
fila = bitacora(
    pregunta="¿Cuál es el efecto promedio del tratamiento en IHDP-medium con nuisance flexibles y sin sesgo de sobreajuste?",
    metodo="DML-PLR con cross-fitting (K=5) y ExtraTrees",
    estimando="ATE (coeficiente PLR)",
    numero_ic=fmt_ci(o["theta"], o["se"]),
    supuesto=supuesto("observable", "medido", f"calidad de los nuisance fuera de muestra: R²(Y~X)={r2_y:.2f}, R²(T~X)={r2_d:.2f}"),
    decision="continuar", lab="L09")

---
## Qué queda para el libro
Ejercicio del libro cubierto: Monte Carlo con y sin cross-fitting y `dml_plr_manual`. Quedan: la derivación de la función de influencia del score (en el libro) y el estudio de la condición de tasa de los nuisance.

*Material formativo (sin nota). Los datos son sintéticos: ver la ficha del escenario.*